In [ ]:
import lsdb
from upath import UPath
from dask.distributed import Client
from lsdb.streams import CatalogStream
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
print(lsdb.__version__)

# Large Crossmatches with EDP2

In this notebook we'll show how to:

* Load the HATS-formatted version of DP2 on the Rubin Science Platform (RSP)
* Load an external HATS catalog
* Apply column-based filtering on catalogs
* Perform large crossmatches between DP2 and other catalogs
* Demonstrate how to use Catalog streaming to overcome significant data volume
* Plot some resulting lightcurves

## Load Data Preview 2

We begin by loading in DP2, specifying just ra and dec columns to have the lightest weight catalog possible. In practice, trimming down columns to only the subset you'll need for analysis can greatly help in avoiding memory-related issues during computation.

> **Tip**: It's fine to load them all lazily at first and then trim down later once you've solidified a workflow.

In [ ]:
dp2_path = UPath("/rubin/lsdb_data/object_collection")

dp2 = lsdb.open_catalog(dp2_path, columns=["coord_ra", "coord_dec"])
dp2

## Load another Catalog - Gaia DR3

We'll now load Gaia remotely as a second catalog for future crossmatching. Again, we're limiting our column loading to only columns we'll need.

Many public catalogs are available to load directly with LSDB — you can browse them at [data.lsdb.io](https://data.lsdb.io).

In [ ]:
# Open from a remote s3 bucket
gaia = lsdb.open_catalog(
    "s3://stpubdata/gaia/gaia_dr3/public/hats",
    columns=[
        "ra",
        "dec",
        "parallax",
        "parallax_over_error",
        "teff_gspphot",
        "logg_gspphot",
        "classprob_dsc_combmod_star",
    ],
)
gaia

### Query Gaia: Filtering for FGK Stars

`Catalog.query` allows us to apply filters based on column values, in this case we'll apply a set of cuts to select for FGK Stars.

In [ ]:
gaia_filtered = gaia.query("parallax > 0 and parallax_over_error > 5 and \
    teff_gspphot > 5380 and teff_gspphot < 7220 and logg_gspphot > 4.5 \
    and logg_gspphot < 4.72 and classprob_dsc_combmod_star > 0.5")

### Perform a Crossmatch

Now we're ready to crossmatch DP2 and our filtered Gaia DR3.

In [ ]:
xmatch = dp2.crossmatch(gaia_filtered, suffix_method="overlapping_columns")
xmatch

## Count the Resulting Matches

At this point, we're just interested in the number of matches we found, so we'll construct a small function to apply through `Catalog.map_partition` which counts the number of matches in each output partition.

In [ ]:
# Write a simple map_partitions function to count resulting matches
def count_matches(df):
    return {"n_matches": len(df)}


matches_by_partition = xmatch.map_partitions(count_matches)

## Performing Computation

For computation, we establish a Dask Client and dictate a number of workers and memory limit per worker. The Dask client will work within it's boundaries to perform the computation that we kick off with `Catalog.compute()` in parallel.

In [ ]:
# Define a client, and .compute() the match result
with Client(n_workers=4, threads_per_worker=1, memory_limit="4GB") as client:
    match_df = matches_by_partition.compute()
    client.close()

Finally we can check the total number of Gaia-filtered FGK stars we matched in DP2:

In [ ]:
print(f"Number of Matches: {match_df["n_matches"].sum()}")

## Including Lightcurves
Let's start again, but this time we'll add the "ObjectForcedSource" column, which includes our lightcurves, with the goal of inspecting a few of them.

In [ ]:
dp2_lc = lsdb.open_catalog(dp2_path, columns=["coord_ra", "coord_dec", "objectForcedSource"])
dp2_lc

A common operation is to filter lightcurve lengths, we lack a pre-existing column to tell us the lengths, so we'll take advantage of our nested column itself to perform the filter:

In [ ]:
# query for lightcurves with at least 50 observations
def filter_lengths(nf):
    # define a function that applies to each dataframe partition
    return nf[nf["objectForcedSource"].len() > 50]  # nested columns have a direct len()


dp2_lc_filtered = dp2_lc.map_partitions(filter_lengths)

Now we can redo our crossmatch with Gaia: 

In [ ]:
xmatch_lc = dp2_lc_filtered.crossmatch(gaia_filtered, suffix_method="overlapping_columns")
xmatch_lc

In [ ]:
# And re-initialize our counting result
def count_matches(df):
    return {"n_matches": len(df)}


matches_by_partition_lc = xmatch_lc.map_partitions(count_matches)

## Dealing with Heavy Workloads - Catalog Streaming

However, our added lightcurves add a ton of weight to the DP2 catalog. If you check the estimated catalog size when we first loaded the catalogs, you can see that DP2 was ~20GB total when loading just the "coord_ra" and "coord_dec" columns, but with "objectForcedSource" this balloons to ~1.3TB. Due to this, it's likely that direct computation on a RSP environment (4 cores, 32GB total) will fail. We can take advantage of an alternative computation method, Catalog Streaming, in cases like this:

In [ ]:
%%time
match_stream = CatalogStream(xmatch_lc, partitions_per_chunk=4, shuffle=False)

chunks = []
# 16 GB workers end up being needed to perform this workflow
with Client(n_workers=1, threads_per_worker=1, memory_limit="16GB") as client:
    print(client)
    match_iter = iter(match_stream)
    for i in range(50):
        chunks.append(next(match_iter))
    client.close()
match_df = pd.concat(chunks)
match_df

Above, we've streamed the result for the first 200 partitions (4 partitions per chunk, 50 chunks grabbed). We ultimately found 11 lightcurves that matched our criteria. Let's end by plotting one of them:

In [ ]:
lc = match_df.iloc[0]["objectForcedSource"]
lc

In [ ]:
plt.plot(lc["midpointMjdTai"], lc["psfMag"], ".")

## Summary

With an RSP-sized environment, it's possible to do large crossmatches between DP2 and other large HATS catalogs, as long as you are mindful of the columns being loaded and the Dask Client setup you are specifying for computation. Note how much the computational approach needed to change just by including lightcurves from "objectForcedSource". Being aware of the computational weight of your workflows is it's own learning curve, even as LSDB tries to assist as much as it can with this.

If you're looking for more tips on doing large scale workflows, consult the [LSDB documentation](https://docs.lsdb.io/en/stable/tutorials/dask-cluster-tips.html).

## About
**Author(s):**Doug Branton

**Last updated on:** July 22, 2026

If you use lsdb for published research, please cite following [instructions](https://docs.lsdb.io/en/stable/citation.html).